# SChain INGEST

Process a file or directory already available in Colab. INGEST does not search the web or
expand the frontier; it runs documents through extraction, resolution, validation and export.

In [ ]:
# Run settings
INPUT_PATH = "/content/documents"
CONFIG_PATH = "config/ingest.yaml"
SAVE_TO_NEO4J = False
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
BRANCH = "mirza"

## Install SChain

In [ ]:
import os
import pathlib
import subprocess
import sys

REPO_DIR = pathlib.Path("/content/SChain")
if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch",
         "https://github.com/OWNER/SChain.git", str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=True)

## Optional Neo4j credentials

In [ ]:
if SAVE_TO_NEO4J:
    from google.colab import userdata

    for name in ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"):
        value = userdata.get(name)
        if not value:
            raise RuntimeError(f"Missing Colab secret: {name}")
        os.environ[name] = value

## Start the local model server

In [ ]:
import httpx
import time

VLLM_URL = "http://127.0.0.1:8000/v1"

def vllm_ready() -> bool:
    try:
        return httpx.get(f"{VLLM_URL}/models", timeout=2).status_code == 200
    except httpx.HTTPError:
        return False

if not vllm_ready():
    log = open("/content/vllm.log", "w", encoding="utf-8")
    subprocess.Popen(
        [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
         "--model", MODEL, "--port", "8000"],
        stdout=log,
        stderr=subprocess.STDOUT,
    )
    for _ in range(120):
        if vllm_ready():
            break
        time.sleep(5)
    else:
        raise RuntimeError("vLLM did not become ready; inspect /content/vllm.log")

## Run INGEST and export the graph

In [ ]:
command = [
    sys.executable, "-m", "schain", "ingest",
    "--config", CONFIG_PATH,
    "--input", INPUT_PATH,
    "--resume",
    "--save" if SAVE_TO_NEO4J else "--no-save",
]
subprocess.run(command, check=True)

subprocess.run([
    sys.executable, "-m", "schain", "export", "--config", CONFIG_PATH,
], check=True)

## Inspect generated artifacts

In [ ]:
import json

for path in sorted(pathlib.Path("outputs").glob("*")):
    print(path)